# Run the processing pipeline

Builds the episode table (Stages 1–3) and runs download, transcription, ad removal and spontaneity check (Stages 4–7).

Run the cells from top to bottom. Transcription costs API credit, so do the smoke test before the full run.
The episode table and the processed table are saved in `data/`, so a later session can continue where the last one stopped.

## 1. Setup

In [11]:
import sys
import logging
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)    # show all columns of a table, never "..."

PROJECT_DIR = Path.cwd().parent
for folder in [PROJECT_DIR, PROJECT_DIR / "preprocessing"]:    # project root holds config.py
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import config
from fetch_pipeline import FetchPipeline
from filter_index import PodcastDatabase
from extract_episodes import EpisodeFetcher
from download import EpisodeDownloader
from transcribe import EpisodeTranscriber
from remove_ads import AdRemover
from classify_spontaneity import SpontaneityClassifier
from processing_pipeline import ProcessingPipeline

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

### Parameters

All settings of a run are in `config.py` in the project folder (years, groups and filters, `K`, `N_PODCASTS`, ad rule, models, paths).
Change them there, not here, and restart the kernel afterwards. This cell only shows the values of this run.

In [ ]:
print(f"Years: {config.YEARS}, screening on {config.SCREENING_YEARS}, 2022 only before {config.CHATGPT_RELEASE:%Y-%m-%d}")
print(f"Groups: {list(config.GROUPS)}")
print(f"N_PODCASTS = {config.N_PODCASTS}, K = {config.K}, K_TEST = {config.K_TEST}, BUFFER = {config.BUFFER}")
print(f"Excerpt: middle {config.DURATION_SEC // 60} minutes of each episode, first {config.MIN_WORDS_PER_EPISODE} words without ads and filler words")

## 2. Build the episode table

Reads the RSS feeds of all matching podcasts, which takes a while. The table is saved and loaded from the file in later sessions.
Delete `data/episodes.pkl` to build it again, for example after changing `GROUPS`, `K`, `BUFFER` or `YEARS` in `config.py`.

In [13]:
if config.EPISODES_PATH.exists():
    episodes = pd.read_pickle(config.EPISODES_PATH)
else:
    tables = []
    for group, filters in config.GROUPS.items():
        pod_db = PodcastDatabase(str(config.DATABASE_PATH), config.DATABASE_TABLE)
        fetch_pipeline = FetchPipeline(pod_db, EpisodeFetcher(), group, filters=filters)
        tables.append(fetch_pipeline.build_podcast_df(k=config.K, buffer=config.BUFFER))
    episodes = pd.concat(tables, ignore_index=True)
    episodes.to_pickle(config.EPISODES_PATH)

print(episodes.groupby("group")["feed_url"].nunique().rename("candidate podcasts per group (pool, the pipeline takes the first N_PODCASTS that pass)"))
episodes.head()

group
first    4
Name: candidate podcasts per group (pool, the pipeline takes the first N_PODCASTS that pass), dtype: int64


,feed_url,podcast_title,episode_title,episode_year,episode_audio_url,episode_length,group,podcast_hash,episode_hash
0,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 77 - Fortnite Fiends,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,3487.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,033a1f839d72626cbdebfbb8e6ed3d5b6e99322124e13e...
1,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 70 | E3 2018 Preshow,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,5537.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,0f79c7a37cb224b49fa85c4c75d51c6886d1f27e961711...
2,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 67 | Is Anthem Doomed?,2018,https://anchor.fm/s/cc813770/podcast/play/6064...,6912.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,135ebed5ae6cb5fce75707ba8ac6ddb9fc38c1fb5387b5...
3,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 74 - The Downfall Of TellTale,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,5106.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,654e432077f230e85ce0bcaad71e44acd20ee3dcc144a1...
4,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 68 | How Fortnite Took Over The World,2018,https://anchor.fm/s/cc813770/podcast/play/6063...,7184.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,6957e1dda5331a6e761f74b44edb5e0f1362decdc87777...


## 3. Create the pipeline

In [14]:
pipeline = ProcessingPipeline(
    ep_downloader=EpisodeDownloader(config.DOWNLOAD_DIR),
    ep_transcriber=EpisodeTranscriber(config.TRANSCRIPTION_DIR),
    ad_remover=AdRemover(config.AD_REMOVED_TRANSCRIPTION_DIR),
    classifier=SpontaneityClassifier(),
)

## 4. Smoke test

One podcast per group with one episode per year. Check the result and the cost before the full run.
Files written here are reused by the full run.

In [ ]:
smoke = pipeline.process(episodes.copy(), k=1, n_podcasts=1, k_test=config.K_TEST)
smoke[smoke["episode_status"] != "pending"][["podcast_title", "podcast_status", "podcast_fail_reason", "episode_year", "episode_status", "episode_fail_reason",
                                             "excerpt_start_sec", "n_words", "n_ad_words", "n_speakers", "score", "scripted"]]

## 5. Full run

Continues from `data/processed.pkl` if it exists, so nothing is processed twice.
If the run is interrupted, run this cell again: finished downloads, transcripts and ad files are reused from disk.

In [16]:
start = pd.read_pickle(config.PROCESSED_PATH) if config.PROCESSED_PATH.exists() else episodes.copy()
before = start["episode_status"].value_counts() if "episode_status" in start.columns else pd.Series({"pending": len(start)})

processed = pipeline.process(start, k=config.K, n_podcasts=config.N_PODCASTS, k_test=config.K_TEST)
processed.to_pickle(config.PROCESSED_PATH)

print("Episodes per status (before -> after this run):")
print(pd.DataFrame({"before": before, "after": processed["episode_status"].value_counts()}).fillna(0).astype(int))

Episodes per status (before -> after this run):
                before  after
episode_status               
pending            596    596
processed           54     54
failed              30     30


## 6. Results

Podcasts per group and status. `pending` podcasts were not needed.

In [17]:
podcasts = processed.drop_duplicates("feed_url")
podcasts.groupby(["group", "podcast_status"]).size().unstack(fill_value=0)

podcast_status,passed,pending
group,,
first,3,1


Processed episodes per podcast and year. Every cell of a passed podcast should hold `K`.

In [18]:
done = processed[(processed["podcast_status"] == "passed") & (processed["episode_status"] == "processed")]
done.groupby(["group", "podcast_title", "episode_year"]).size().unstack(fill_value=0)

episode_year                2018  2019  2020  2021  2022  2023  2024  2025  \
group podcast_title                                                          
first Gaming illuminaughty     2     2     2     2     2     2     2     2   
      Rand(Nerds);             2     2     2     2     2     2     2     2   
      Three Moves Ahead        2     2     2     2     2     2     2     2   

episode_year                2026  
group podcast_title               
first Gaming illuminaughty     2  
      Rand(Nerds);             2  
      Three Moves Ahead        2

All columns of the processed episodes (first rows). Scroll sideways to see everything.
The second table shows one episode with its columns as rows, which is easier to read.

In [25]:
done.tail(50)

,feed_url,podcast_title,episode_title,episode_year,episode_audio_url,episode_length,group,podcast_hash,episode_hash,episode_status,fail_stage,episode_fail_reason,audio_path,transcript_path,cleaned_path,n_words,n_speakers,main_speaker_share,mean_confidence,score,scripted,scripted_reason,llm_model,podcast_status,podcast_fail_reason
28,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 104 - Wakanda Forever,2020,https://anchor.fm/s/cc813770/podcast/play/6062...,5125.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,0193ca44c097d889432bcaea97bb1c625e4160619eda8a...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2304,2,0.658854,0.953595,4.0,False,"The dialogue contains many filler words, self‑...",openai/gpt-oss-120b,passed,<NA>
29,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 105 - Smoking On A Nuggets Pack,2020,https://anchor.fm/s/cc813770/podcast/play/6062...,4306.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,0d67edcd242082269d87e16ad7c0a446bcdf5c83e13671...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2353,4,0.503187,0.950258,4.0,False,"The dialogue contains many filler words, hesit...",openai/gpt-oss-120b,passed,<NA>
44,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 119 - NFT's Invade Gaming,2021,https://anchor.fm/s/cc813770/podcast/play/6062...,6183.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,002898a7c6347b4b8614bea05e38ed84c3148bb9bf3608...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2255,4,0.354324,0.923379,4.0,False,"The speakers use many filler words, repetition...",openai/gpt-oss-120b,passed,<NA>
45,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 116 - Halo Infinite Possibilities,2021,https://anchor.fm/s/cc813770/podcast/play/6062...,5258.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,51ddf93ee35767964ba8508b9909c14989e943b0e77fda...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2371,4,0.548714,0.937807,4.0,False,"Speaker 3 uses many filler words, repetitions,...",openai/gpt-oss-120b,passed,<NA>
53,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 132 - Saints Row Let Nas Down,2022,https://anchor.fm/s/cc813770/podcast/play/6062...,5306.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,17f1d33d9dd702bfd9c7fe092f44909ebbb017bbd5154f...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2234,2,0.690242,0.959923,4,False,"The dialogue contains many filler words, hesit...",openai/gpt-oss-120b,passed,<NA>
54,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 137 - Top 5 Games Of 2022,2022,https://anchor.fm/s/cc813770/podcast/play/6291...,7495.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,20cb6d1018b212a95c8d38e787be484424ea2fec68e606...,processed,<NA>,<NA>,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,/Users/lucasfreigang/PSL/Behavioral_Science_Ap...,2419,4,0.577925,0.90973,4.0,False,"The dialogue is filled with hesitations, fille...",openai/gpt-oss-120b,passed,<NA>
71,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 142 - Redfall or RedFAIL?!,2023,https://anchor.fm/s/cc813770/podcast/play/7045...,6953.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,0002f7d77d57ae0cddb5157e35ab3cbcab04c4ca

In [20]:
done.iloc[0].to_frame("first processed episode")

,first processed episode
feed_url,https://feeds.feedburner.com/GamingIlluminaugh...
podcast_title,Gaming illuminaughty
episode_title,Episode 77 - Fortnite Fiends
episode_year,2018
episode_audio_url,https://anchor.fm/s/cc813770/podcast/play/6062...
episode_length,3487.0
group,first
podcast_hash,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...
episode_hash,033a1f839d72626cbdebfbb8e6ed3d5b6e99322124e13e...
episode_status,processed


Why podcasts and episodes failed.

In [21]:
print(podcasts[podcasts["podcast_status"] == "failed"]["podcast_fail_reason"].value_counts(), "\n")

failed = processed[processed["episode_status"] == "failed"]
print(failed["fail_stage"].value_counts())
failed[["podcast_title", "episode_year", "fail_stage", "episode_fail_reason"]]

Series([], Name: count, dtype: int64) 

fail_stage
spontaneity    30
Name: count, dtype: int64


,podcast_title,episode_year,fail_stage,episode_fail_reason
83,Gaming illuminaughty,2024,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
96,Gaming illuminaughty,2025,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
104,Gaming illuminaughty,2026,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
105,Gaming illuminaughty,2026,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
175,Rand(Nerds);,2019,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
176,Rand(Nerds);,2019,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
200,Rand(Nerds);,2020,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
227,Rand(Nerds);,2021,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
278,Rand(Nerds);,2023,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
279,Rand(Nerds);,2023,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."


Spontaneity score and transcription quality per group and year (processed episodes of passed podcasts).

In [22]:
done.astype({"score": float, "mean_confidence": float, "n_words": float}).groupby(["group", "episode_year"])[["score", "mean_confidence", "n_words"]].mean().round(2)

score  mean_confidence  n_words
group episode_year                                 
first 2018            4.0             0.93  1925.67
      2019            4.0             0.93  2088.17
      2020            4.0             0.94  1977.33
      2021            4.0             0.93  1840.83
      2022            4.0             0.94  1991.50
      2023            4.0             0.92  1958.50
      2024            4.0             0.94  2058.17
      2025            4.0             0.94  2086.33
      2026            4.0             0.95  2087.50